# Exp-5 — Case-first retrieval over `court_considerations.csv` + statute mining

**Why this exp.** All previous experiments (Exp-1 through Exp-4b) only retrieved from the statute corpus and topped out at `stat_recall@500 ≈ 0.376`. Meanwhile **40% of val gold is case-law citations** (27.5% BGE + 13.1% docket) — and *no experiment has touched the court corpus*. By construction, macro-F1 is capped below 0.5 until we retrieve cases.

**Two coupled moves in one notebook:**

**A5 — Case retrieval.** Encode all 2.47M `court_considerations.csv` rows with BGE-M3 (one unified DE/FR/IT index — *no* language filter; agent research says splitting per-language costs cross-lingual recall). Retrieve top-K cases per query using the Exp-3 `q_hyde + q_enum + q_de` RRF query vectors. Eval: `case_recall@{50,100,200,500}` against val BGE-and-docket gold.

**A5b — Inline statute mining.** Run a trilingual regex (DE/FR/IT) over the top-K retrieved case text → extract `Art. N ABBR` mentions → normalise FR/IT abbreviations (CO↔OR, CC↔ZGB, CP↔StGB, CPP↔StPO, CPC↔ZPO, LTF↔BGG) → look up against `laws_de.citation` set. This is the 'lawyer's reasoning loop': find the case, then read off its cited articles. Reports `mined_stat_recall@K` against val statute gold.

**Gates.**
- `case_recall@200 ≥ 0.30`: confirms the case branch is producing real signal (currently 0.0 since untouched).
- `mined_stat_recall@200 ≥ 0.20`: confirms statute mining from cases adds non-trivial recall on top of the existing pipeline (Exp-3 alone is 0.315 @200).
- Combined `(stat_via_cases ∪ exp3_stat) ≥ 0.45`: confirms statute retrieval breaks past the 0.376 ceiling via case-corpus signal.

**Resumability.** Encoding 2.47M rows takes ~1–4 hrs depending on GPU. We chunk the corpus (100K rows per chunk) and save each chunk's embeddings to `artifacts/court_chunks/court_NNNN.npy` so a runtime disconnect can resume mid-encode.

**Storage.** 2,476,315 × 1024 × fp16 ≈ **5.06 GB** for the final embedding file. Fits in Drive.

In [1]:
# --- Cell 1. Install deps (matches Exp-4b layout: numpy 1.26 first, then FlagEmbedding) ---
!pip install -qU numpy==1.26.4
!pip install -qU FlagEmbedding pandas
# Restart runtime if numpy was downgraded — re-run from Cell 2 after restart.

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.0/61.0 kB 5.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.0/18.0 MB 181.2 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
tobler 0.14.0 requires numpy>=2.0, but you have numpy 1.26.4 which is incompatible.
rasterio 1.5.0 requires numpy>=2, but you have numpy 1.26.4 which is incompatible.
shap 0.51.0 requires numpy>=2, but you have numpy 1.26.4 which is incompatible.
xarray-einstats 0.10.0 requires numpy>=2.0, but you have numpy 1.26.4 which is incompatible.
opencv-python-headless 4.13.0.92 requires numpy>=2; python_version >= "3.9", but you have numpy 1.26.4 which is incompatible.
jaxlib 0.7.2 requires numpy>=2.0, but you have numpy 1.26.4 which is incompatible.
pytensor 2.38.2 requires numpy>=2.0, but you have numpy 1.26.4 which is incompatible.
jax 0.7.2 requires numpy>=2.0,

In [4]:
# --- Cell 1. Install deps (Qwen via vLLM for fast batched inference) ---
!pip install -qU FlagEmbedding

In [5]:
!pip install -qU vllm

In [4]:
!pip install -qU "transformers<4.45.0"

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.7/43.7 kB 3.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.5/9.5 MB 46.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 66.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 34.0 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
vllm 0.19.1 requires tokenizers>=0.21.1, but you have tokenizers 0.19.1 which is incompatible.
vllm 0.19.1 requires transformers!=5.0.*,!=5.1.*,!=5.2.*,!=5.3.*,!=5.4.*,!=5.5.0,>=4.56.0, but you have transformers 4.44.2 which is incompatible.
compressed-tensors 0.15.0.1 requires transformers>=4.45.0, but you have transformers 4.44.2 which is incompatible.
gradio 5.50.0 requires pandas<3.0,>=1.0, but you have pandas 3.0.2 which is incompatible.


In [1]:
# --- Cell 2. Mount Drive, paths, GPU check ---
from google.colab import drive
drive.mount('/content/drive')

import json, pickle, re, time, gc
from pathlib import Path
import numpy as np
import pandas as pd
import torch

ROOT = Path('/content/drive/MyDrive/swiss_law/data')
ART  = ROOT / 'artifacts'
CHUNKS_DIR = ART / 'court_chunks'
CHUNKS_DIR.mkdir(parents=True, exist_ok=True)

for f in ['court_considerations.csv', 'val.csv', 'laws_de.csv']:
    assert (ROOT / f).exists(), f'missing {f} on Drive'
for f in ['query_vecs_A3.npz', 'exp_A3_expansions.json']:
    assert (ART / f).exists(), f'missing {f} — run Exp-3 first'

assert torch.cuda.is_available(), 'need GPU'
print('GPU:', torch.cuda.get_device_name(0))
print('Free VRAM (GB):', round(torch.cuda.mem_get_info()[0] / 1e9, 1))

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
GPU: NVIDIA RTX PRO 6000 Blackwell Server Edition
Free VRAM (GB): 101.4


In [2]:
# --- Cell 3. Load val + Exp-3 query vectors + laws_de set (for mining lookup) ---
val  = pd.read_csv(ROOT / 'val.csv')
qv = np.load(ART / 'query_vecs_A3.npz')
q_en, q_de, q_hyde, q_enum = qv['q_en'], qv['q_de'], qv['q_hyde'], qv['q_enum']
query_ids = [str(x) for x in qv['query_ids']]
assert query_ids == list(val['query_id']), 'order mismatch'

laws = pd.read_csv(ROOT / 'laws_de.csv')
laws_cits = set(laws['citation'].astype(str))
# also build a prefix lookup: 'Art. 100 BGG' -> [full citations starting with that prefix]
from collections import defaultdict
laws_by_prefix = defaultdict(list)
PREFIX_PAT = re.compile(r'^(Art\.?\s*\d+[a-z]?\s+[A-ZÄÖÜ]+\b)')
for c in laws_cits:
    m = PREFIX_PAT.match(c)
    if m:
        laws_by_prefix[m.group(1).replace('Art ', 'Art. ')].append(c)
print('val:', len(val), '| laws:', len(laws), '| unique law prefixes:', len(laws_by_prefix))
print('q_hyde:', q_hyde.shape, '| q_enum:', q_enum.shape, '| q_de:', q_de.shape)

val: 10 | laws: 175933 | unique law prefixes: 9594
q_hyde: (10, 1024) | q_enum: (10, 1024) | q_de: (10, 1024)


In [3]:
# --- Cell 4. Load BGE-M3 (skip if final embedding already cached) ---
FINAL_EMB = ART / 'court_bgem3.npy'
CITS_JSON = ART / 'court_citations.json'

if FINAL_EMB.exists() and CITS_JSON.exists():
    print('cached corpus found — skip Cells 4-7, jump to Cell 8')
    model = None
else:
    from FlagEmbedding import BGEM3FlagModel
    model = BGEM3FlagModel('BAAI/bge-m3', use_fp16=True)
    print('BGE-M3 loaded')

tokenizer_config.json:   0%|          | 0.00/444 [00:00<?, ?B/s]

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/17.1M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/964 [00:00<?, ?B/s]

Fetching 30 files:   0%|          | 0/30 [00:00<?, ?it/s]

...

onnx/tokenizer.json:   0%|          | 0.00/17.1M [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/54.0 [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/2.27G [00:00<?, ?B/s]

sparse_linear.pt:   0%|          | 0.00/3.52k [00:00<?, ?B/s]

BGE-M3 loaded


In [4]:
# --- Cell 5. Encode court_considerations in resumable chunks ---
# Strategy: stream the CSV in chunks of 100k rows; one .npy per chunk; resume by skipping existing files.
# At ~250–800 docs/sec (T4 → A100), 2.47M rows take 1–4 hrs.

CHUNK_SIZE = 100_000
BATCH = 64
MAXLEN = 512  # short considerations dominate; this caps cost without losing much signal

if FINAL_EMB.exists() and CITS_JSON.exists():
    print('skipping encode — final embeddings already cached')
else:
    all_cits_in_order = []
    t0 = time.time()
    for chunk_id, df in enumerate(pd.read_csv(ROOT / 'court_considerations.csv',
                                              chunksize=CHUNK_SIZE)):
        all_cits_in_order.extend(df['citation'].fillna('').astype(str).tolist())
        out_path = CHUNKS_DIR / f'court_{chunk_id:04d}.npy'
        if out_path.exists():
            print(f'  chunk {chunk_id:04d} ({len(df)} rows) — already done')
            continue
        docs = df['text'].fillna('').astype(str).tolist()
        embs = model.encode(docs, batch_size=BATCH, max_length=MAXLEN,
                            return_dense=True, return_sparse=False,
                            return_colbert_vecs=False)['dense_vecs'].astype(np.float16)
        np.save(out_path, embs)
        elapsed = time.time() - t0
        done = chunk_id + 1
        print(f'  chunk {chunk_id:04d}: {len(df)} rows | elapsed {elapsed/60:.1f} min '
              f'| pace {(done * CHUNK_SIZE)/elapsed:.0f} docs/sec')

    with open(CITS_JSON, 'w') as f:
        json.dump(all_cits_in_order, f)
    print(f'done all chunks in {(time.time()-t0)/60:.1f} min '
          f'| {len(all_cits_in_order)} citations saved')

pre tokenize: 100%|██████████| 1563/1563 [00:09<00:00, 165.20it/s]
You're using a XLMRobertaTokenizerFast tokenizer. Please note that with a fast tokenizer, using the `__call__` method is faster than using a method to encode the text followed by a call to the `pad` method to get a padded encoding.
Inference Embeddings: 100%|██████████| 1563/1563 [01:57<00:00, 13.27it/s] 


  chunk 0000: 100000 rows | elapsed 2.2 min | pace 748 docs/sec


Inference Embeddings: 100%|██████████| 1563/1563 [01:19<00:00, 19.57it/s] 


  chunk 0001: 100000 rows | elapsed 3.7 min | pace 903 docs/sec


Inference Embeddings: 100%|██████████| 1563/1563 [01:18<00:00, 19.92it/s] 


...

  chunk 0022: 100000 rows | elapsed 36.6 min | pace 1049 docs/sec


Inference Embeddings: 100%|██████████| 1563/1563 [01:33<00:00, 16.74it/s] 


  chunk 0023: 100000 rows | elapsed 38.3 min | pace 1045 docs/sec


Inference Embeddings: 100%|██████████| 1193/1193 [01:03<00:00, 18.66it/s] 


  chunk 0024: 76315 rows | elapsed 39.5 min | pace 1056 docs/sec
done all chunks in 39.5 min | 2476315 citations saved


In [5]:
# --- Cell 6. Concatenate chunks → single court_bgem3.npy (5 GB fp16) ---
if FINAL_EMB.exists():
    print('final embedding already exists — skip')
else:
    chunk_files = sorted(CHUNKS_DIR.glob('court_*.npy'))
    print(f'concatenating {len(chunk_files)} chunks…')
    parts = [np.load(f) for f in chunk_files]
    court_emb = np.concatenate(parts, axis=0)
    print('concatenated:', court_emb.shape, court_emb.dtype,
          f'({court_emb.nbytes/1e9:.2f} GB)')
    np.save(FINAL_EMB, court_emb)
    del parts; gc.collect()
    print('saved:', FINAL_EMB)

concatenating 25 chunks…
concatenated: (2476315, 1024) float16 (5.07 GB)
saved: /content/drive/MyDrive/swiss_law/data/artifacts/court_bgem3.npy


In [6]:
# --- Cell 7. Free LLM weights, load embeddings + citation list ---
if model is not None:
    del model; gc.collect(); torch.cuda.empty_cache()

court_emb = np.load(FINAL_EMB, mmap_mode='r')   # mmap so we don't OOM on RAM
with open(CITS_JSON, 'r') as f:
    court_cits = json.load(f)
assert len(court_cits) == court_emb.shape[0], (len(court_cits), court_emb.shape)
print('court corpus:', court_emb.shape, court_emb.dtype, '| citations:', len(court_cits))

court corpus: (2476315, 1024) float16 | citations: 2476315


In [8]:
# --- Cell 8. Top-K retrieval over court corpus (chunked GPU matmul) ---
# Test 4 query forms: q_de, q_hyde, q_enum, RRF(hyde,enum,de)
TOPK = 500

def topk_court_gpu(q_vecs, court_emb, k=TOPK, chunk=200_000):
    N_q, D = q_vecs.shape; N_doc = court_emb.shape[0]
    qg = torch.from_numpy(q_vecs.astype(np.float16)).cuda()
    # Fix: Use a value within float16 range instead of -1e9
    top_scores = torch.full((N_q, k), -60000.0, dtype=torch.float16, device='cuda')
    top_idx    = torch.full((N_q, k), -1,    dtype=torch.int64,  device='cuda')
    for s in range(0, N_doc, chunk):
        e = min(s + chunk, N_doc)
        block = torch.from_numpy(np.asarray(court_emb[s:e])).cuda()  # (chunk, D) fp16
        sc = qg @ block.T                                            # (N_q, chunk)
        merged_sc  = torch.cat([top_scores, sc], dim=1)
        merged_idx = torch.cat([top_idx,
                                torch.arange(s, e, device='cuda').unsqueeze(0).expand(N_q, -1)],
                               dim=1)
        v, ord_ = torch.topk(merged_sc, k, dim=1)
        top_scores = v
        top_idx    = torch.gather(merged_idx, 1, ord_)
        del block, sc, merged_sc, merged_idx
    return top_idx.cpu().numpy(), top_scores.float().cpu().numpy()

def rank_to_pos(idx_topk, N_doc):
    """Convert top-k indices to a full-rank position array (for RRF)."""
    N_q, K = idx_topk.shape
    pos = np.full((N_q, N_doc), K, dtype=np.int32)  # unranked = pos K (worst rank+1)
    rows = np.arange(N_q)[:, None]
    pos[rows, idx_topk] = np.arange(K)[None, :]
    return pos

t0 = time.time()
idx_de,   _ = topk_court_gpu(q_de,   court_emb)
print(f'q_de done in {time.time()-t0:.0f}s')
t0 = time.time()
idx_hyde, _ = topk_court_gpu(q_hyde, court_emb)
print(f'q_hyde done in {time.time()-t0:.0f}s')
t0 = time.time()
idx_enum, _ = topk_court_gpu(q_enum, court_emb)
print(f'q_enum done in {time.time()-t0:.0f}s')


/tmp/ipykernel_11507/1581485073.py:13: UserWarning: The given NumPy array is not writable, and PyTorch does not support non-writable tensors. This means writing to this tensor will result in undefined behavior. You may want to copy the array to protect its data or make it writable before converting it to a tensor. This type of warning will be suppressed for the rest of this program. (Triggered internally at /pytorch/torch/csrc/utils/tensor_numpy.cpp:213.)
  block = torch.from_numpy(np.asarray(court_emb[s:e])).cuda()  # (chunk, D) fp16


q_de done in 1s
q_hyde done in 0s
q_enum done in 0s


In [9]:
# --- Cell 9. RRF fusion of the 3 query forms ---
def rrf_fuse(top_idx_list, k_rrf=60, k_out=TOPK):
    N_q = top_idx_list[0].shape[0]
    cand_scores = [{} for _ in range(N_q)]
    for top_idx in top_idx_list:
        for q in range(N_q):
            for rank, doc in enumerate(top_idx[q]):
                cand_scores[q][int(doc)] = cand_scores[q].get(int(doc), 0.0) + 1.0/(k_rrf + rank)
    out = np.full((N_q, k_out), -1, dtype=np.int64)
    for q in range(N_q):
        ranked = sorted(cand_scores[q].items(), key=lambda kv: -kv[1])[:k_out]
        for r,(doc,_) in enumerate(ranked):
            out[q, r] = doc
    return out

idx_rrf = rrf_fuse([idx_de, idx_hyde, idx_enum])
print('RRF top-K:', idx_rrf.shape)

RRF top-K: (10, 500)


In [10]:
# --- Cell 10. Case-recall evaluation ---
def parse(s): return [c.strip() for c in str(s).split(';') if c.strip()]
def cit_class(c):
    if c.startswith('BGE '):    return 'bge'
    if re.match(r'^\d[A-Z]_', c) or re.match(r'^[A-Z]\d[A-Z]_', c): return 'docket'
    return 'statute'

DOCKET_RE = re.compile(r'^([1-9][A-Z]?_\d+/\d{4})')

def case_match(gold, corpus_cit):
    """True if the gold case citation matches the corpus citation.
    Handles BGE 'BGE V P R E. N' (gold) vs 'BGE V P R E. N(.subN)' (corpus),
    and docket gold (e.g. '4A_123/2020') vs corpus 'docket-prefixed' rows."""
    if not gold or not corpus_cit: return False
    g, c = gold.strip(), corpus_cit.strip()
    if g == c: return True
    if g.startswith('BGE '):
        # corpus may add a sub-numbered Erwägung after gold's E. N
        return c.startswith(g)
    # docket: gold = '4A_123/2020', corpus may = '4A_123/2020' or '4A_123/2020 E. 2.3' or longer
    md = DOCKET_RE.match(g)
    if md:
        return c.startswith(md.group(1))
    return False

def eval_case_recall(top_idx, label, ks=(50,100,200,500)):
    per_q, agg_num, agg_den = [], {k:0 for k in ks}, 0
    for i, row in enumerate(val.itertuples()):
        gold = [c for c in parse(row.gold_citations) if cit_class(c) in ('bge','docket')]
        agg_den += len(gold)
        cits_at = {k: [court_cits[j] for j in top_idx[i, :k]] for k in ks}
        entry = {'query_id': row.query_id, 'n_case_gold': len(gold)}
        for k in ks:
            hit = sum(1 for g in gold if any(case_match(g, cc) for cc in cits_at[k]))
            entry[f'hit@{k}'] = hit
            agg_num[k] += hit
        per_q.append(entry)
    agg = {f'case_recall@{k}': agg_num[k]/max(1,agg_den) for k in ks}
    print(f'=== {label} ===')
    for k,v in agg.items(): print(f'  {k} = {v:.3f}')
    return {'agg': agg, 'per_query': per_q}

case_eval = {
    'q_de':   eval_case_recall(idx_de,   'q_de'),
    'q_hyde': eval_case_recall(idx_hyde, 'q_hyde'),
    'q_enum': eval_case_recall(idx_enum, 'q_enum'),
    'rrf':    eval_case_recall(idx_rrf,  'RRF(de,hyde,enum)'),
}

=== q_de ===
  case_recall@50 = 0.010
  case_recall@100 = 0.020
  case_recall@200 = 0.049
  case_recall@500 = 0.147
=== q_hyde ===
  case_recall@50 = 0.059
  case_recall@100 = 0.069
  case_recall@200 = 0.098
  case_recall@500 = 0.176
=== q_enum ===
  case_recall@50 = 0.029
  case_recall@100 = 0.049
  case_recall@200 = 0.059
  case_recall@500 = 0.108
=== RRF(de,hyde,enum) ===
  case_recall@50 = 0.049
  case_recall@100 = 0.059
  case_recall@200 = 0.108
  case_recall@500 = 0.157


In [11]:
# --- Cell 11. A5b — Trilingual statute mining from retrieved case text ---
# We stream the CSV again, but only read the rows we actually need (the union of indices across queries).
MINE_K = 200  # mine over the top-200 cases per query
needed_rows = sorted(set(int(j) for q in range(idx_rrf.shape[0]) for j in idx_rrf[q, :MINE_K]))
needed_set = set(needed_rows)
print(f'reading text for {len(needed_rows)} unique case rows…')

row_text = {}
rows_seen = 0
for chunk_id, df in enumerate(pd.read_csv(ROOT / 'court_considerations.csv', chunksize=100_000)):
    base = chunk_id * 100_000
    for local_i, txt in enumerate(df['text'].fillna('').astype(str).tolist()):
        gi = base + local_i
        if gi in needed_set:
            row_text[gi] = txt
    rows_seen += len(df)
    if len(row_text) >= len(needed_set):
        break
print(f'collected text for {len(row_text)} rows')

STATUTE_DE = re.compile(
    r'Art\.?\s*(\d+[a-z]?)(?:\s*Abs\.?\s*\d+)?(?:\s*(?:lit\.?|Bst\.?)\s*[a-z])?'
    r'(?:\s*Ziff\.?\s*\d+)?\s+'
    r'(ZGB|OR|StGB|StPO|ZPO|BGG|BV|URG|MSchG|DBG|MWSTG|SchKG|AHVG|IVG|UVG|KVG|'
    r'BG\u00d6|DSG|AIG|AuG|AsylG|VZV|SVG|VStrR|VStG|MStG|TSchG|RPG|FINIG|FINFRAG)\b'
)
STATUTE_FR = re.compile(
    r'art\.?\s*(\d+[a-z]?)(?:\s*al\.?\s*\d+)?(?:\s*let\.?\s*[a-z])?\s+'
    r'(CO|CC|CP|CPP|CPC|LTF|Cst|LDIP|LP)\b', re.IGNORECASE)
STATUTE_IT = re.compile(
    r'art\.?\s*(\d+[a-z]?)(?:\s*cpv\.?\s*\d+)?(?:\s*lett\.?\s*[a-z])?\s+'
    r'(CO|CC|CP|CPP|CPC|LTF|Cost|LDIP|LP)\b', re.IGNORECASE)
FR2DE = {'CO':'OR','CC':'ZGB','CP':'StGB','CPP':'StPO','CPC':'ZPO','LTF':'BGG',
         'CST':'BV','LDIP':'IPRG','LP':'SchKG'}
IT2DE = FR2DE | {'COST':'BV'}

def mine(text):
    """Yield canonical 'Art. N ABBR' (German) statute citations found in text."""
    found = set()
    for m in STATUTE_DE.finditer(text):
        found.add(f'Art. {m.group(1)} {m.group(2)}')
    for m in STATUTE_FR.finditer(text):
        abbr = FR2DE.get(m.group(2).upper(), m.group(2).upper())
        found.add(f'Art. {m.group(1)} {abbr}')
    for m in STATUTE_IT.finditer(text):
        abbr = IT2DE.get(m.group(2).upper(), m.group(2).upper())
        found.add(f'Art. {m.group(1)} {abbr}')
    return found

# For each query, walk top-K cases (in rank order) and accumulate mined statutes.
# We keep first-seen rank as the candidate's rank for downstream evaluation.
def mined_statutes_per_query(top_idx, k):
    out = []
    for q in range(top_idx.shape[0]):
        ranked = []
        seen = set()
        for j in top_idx[q, :k]:
            txt = row_text.get(int(j), '')
            for prefix in mine(txt):
                if prefix in seen: continue
                seen.add(prefix)
                # expand prefix → all matching laws_de citations (recall over Abs./lit. variants)
                for full in laws_by_prefix.get(prefix, []):
                    ranked.append(full)
                # also keep bare prefix itself if it exists in laws set
                if prefix in laws_cits and prefix not in ranked:
                    ranked.append(prefix)
        out.append(ranked)
    return out

mined_50  = mined_statutes_per_query(idx_rrf, 50)
mined_100 = mined_statutes_per_query(idx_rrf, 100)
mined_200 = mined_statutes_per_query(idx_rrf, 200)
print('mined sizes (q0):', len(mined_50[0]), len(mined_100[0]), len(mined_200[0]))

reading text for 1934 unique case rows…
collected text for 1934 rows
mined sizes (q0): 0 4 7


In [12]:
# --- Cell 12. Mined-statute recall vs Exp-3 baseline ---
def eval_mined_stat(mined_per_q, label):
    num, den = 0, 0
    per_q = []
    for i, row in enumerate(val.itertuples()):
        gold = {c for c in parse(row.gold_citations) if cit_class(c)=='statute'}
        den += len(gold)
        hit = len(gold & set(mined_per_q[i]))
        num += hit
        per_q.append({'query_id': row.query_id, 'n_stat_gold': len(gold),
                      'mined_n': len(mined_per_q[i]), 'hit': hit})
    rec = num / max(1,den)
    print(f'{label}: mined_stat_recall = {rec:.3f}  ({num}/{den})')
    return {'recall': rec, 'per_query': per_q}

mine_eval = {
    'mine_top50_cases':  eval_mined_stat(mined_50,  'mine over top-50 cases'),
    'mine_top100_cases': eval_mined_stat(mined_100, 'mine over top-100 cases'),
    'mine_top200_cases': eval_mined_stat(mined_200, 'mine over top-200 cases'),
}

# Combined: union of mined statutes (top-200 cases) + Exp-3 hyde+enum statute candidates (top-200)
# Reuse the candidates from rerank_scores_A4.npz if available, else just compare mined alone.
try:
    saved_a4 = np.load(ART / 'rerank_scores_A4.npz', allow_pickle=True)
    cands_stat = saved_a4['cands']  # (10, 1000) statute indices
    laws_cits_list = laws['citation'].tolist()
    union_recall_at = {}
    for K_stat in (50, 100, 200, 500):
        num, den = 0, 0
        for i, row in enumerate(val.itertuples()):
            gold = {c for c in parse(row.gold_citations) if cit_class(c)=='statute'}
            stat_top = {laws_cits_list[j] for j in cands_stat[i, :K_stat]}
            mined = set(mined_200[i])
            den += len(gold)
            num += len(gold & (stat_top | mined))
        union_recall_at[K_stat] = num/max(1,den)
        print(f'combined stat_recall(stat_top@{K_stat} ∪ mined@200_cases) = {num/max(1,den):.3f}')
except FileNotFoundError:
    print('no rerank_scores_A4.npz found — combined-recall comparison skipped')
    union_recall_at = {}

mine over top-50 cases: mined_stat_recall = 0.047  (7/149)
mine over top-100 cases: mined_stat_recall = 0.047  (7/149)
mine over top-200 cases: mined_stat_recall = 0.060  (9/149)
combined stat_recall(stat_top@50 ∪ mined@200_cases) = 0.215
combined stat_recall(stat_top@100 ∪ mined@200_cases) = 0.302
combined stat_recall(stat_top@200 ∪ mined@200_cases) = 0.349
combined stat_recall(stat_top@500 ∪ mined@200_cases) = 0.409


In [13]:
# --- Cell 13. Per-query breakdown ---
best_case = case_eval['rrf']['per_query']
best_mine = mine_eval['mine_top200_cases']['per_query']
print(f'{"qid":<10} {"caseG":>5} {"case@50":>8} {"case@200":>8} | '
      f'{"statG":>5} {"mineN":>5} {"mineHit":>7}')
print('-'*70)
for i, row in enumerate(val.itertuples()):
    c = best_case[i]; m = best_mine[i]
    print(f'{row.query_id:<10} {c["n_case_gold"]:>5} {c["hit@50"]:>8} {c["hit@200"]:>8} | '
          f'{m["n_stat_gold"]:>5} {m["mined_n"]:>5} {m["hit"]:>7}')

qid        caseG  case@50 case@200 | statG mineN mineHit
----------------------------------------------------------------------
val_001       23        0        2 |    19     7       0
val_002       16        0        1 |    20     2       0
val_003       23        0        0 |    24     6       0
val_004        1        0        1 |     9    18       3
val_005        5        2        2 |     6    12       0
val_006        7        1        2 |    11    16       0
val_007        4        0        0 |    15    27       2
val_008        9        0        0 |    20    14       2
val_009        3        0        0 |    11    14       1
val_010       11        2        3 |    14     8       1


In [14]:
# --- Cell 14. Save artifacts + verdict ---
report = {
    'case_retrieval': {k: v['agg'] for k,v in case_eval.items()},
    'case_per_query': {k: v['per_query'] for k,v in case_eval.items()},
    'statute_mining': {k: v['recall'] for k,v in mine_eval.items()},
    'mining_per_query': {k: v['per_query'] for k,v in mine_eval.items()},
    'combined_stat_recall_union': union_recall_at,
    'meta': {
        'corpus': 'court_considerations.csv (2.47M rows, unified DE/FR/IT)',
        'encoder': 'BAAI/bge-m3 (fp16, max_len=512)',
        'query_forms': ['q_de','q_hyde','q_enum','RRF(de,hyde,enum)'],
        'mining_regex': 'trilingual DE/FR/IT, FR/IT abbrevs normalised to DE',
        'gates': {
            'case_recall@200_floor': 0.30,
            'mined_stat_recall@200_useful': 0.20,
            'combined_stat_recall_target': 0.45,
        }
    }
}
with open(ART / 'exp_A5_report.json', 'w') as f:
    json.dump(report, f, indent=2, default=str)
np.savez(ART / 'court_topk_A5.npz',
         idx_de=idx_de, idx_hyde=idx_hyde, idx_enum=idx_enum, idx_rrf=idx_rrf,
         query_ids=np.array(query_ids))

print('='*70)
print(f'CASE RECALL  (best: RRF)')
for k,v in case_eval['rrf']['agg'].items():
    print(f'  {k} = {v:.3f}')
print('STATUTE MINING (top-200 cases)')
print(f'  mined_stat_recall = {mine_eval["mine_top200_cases"]["recall"]:.3f}')
if union_recall_at:
    print('COMBINED STAT (Exp-3 candidates ∪ mined)')
    for k,v in union_recall_at.items(): print(f'  combined@stat_top{k} ∪ mined@200 = {v:.3f}')

case_pass = case_eval['rrf']['agg']['case_recall@200'] >= 0.30
mine_pass = mine_eval['mine_top200_cases']['recall'] >= 0.20
comb_pass = max(union_recall_at.values()) >= 0.45 if union_recall_at else False
print(f'\ngate case_recall@200 ≥ 0.30  -> {"PASS" if case_pass else "FAIL"}')
print(f'gate mined_stat_recall ≥ 0.20 -> {"PASS" if mine_pass else "FAIL"}')
print(f'gate combined_stat ≥ 0.45    -> {"PASS" if comb_pass else "FAIL"}')

CASE RECALL  (best: RRF)
  case_recall@50 = 0.049
  case_recall@100 = 0.059
  case_recall@200 = 0.108
  case_recall@500 = 0.157
STATUTE MINING (top-200 cases)
  mined_stat_recall = 0.060
COMBINED STAT (Exp-3 candidates ∪ mined)
  combined@stat_top50 ∪ mined@200 = 0.215
  combined@stat_top100 ∪ mined@200 = 0.302
  combined@stat_top200 ∪ mined@200 = 0.349
  combined@stat_top500 ∪ mined@200 = 0.409

gate case_recall@200 ≥ 0.30  -> FAIL
gate mined_stat_recall ≥ 0.20 -> FAIL
gate combined_stat ≥ 0.45    -> FAIL
